# Build and ship a reviewed revision

Create a tiny local artifact, produce an immutable SHA-256 manifest, and assemble a fail-closed publishing checklist. This notebook does not upload, submit, or publish anything.

In [ ]:
import hashlib
import json
import mimetypes
import tempfile
from pathlib import Path

workspace_handle = tempfile.TemporaryDirectory(prefix="superii-revision-")
workspace = Path(workspace_handle.name)
(workspace / "README.md").write_text("# Example revision\n\nSynthetic tutorial artifact.\n", encoding="utf-8")
(workspace / "config.json").write_text(json.dumps({"format": "tutorial", "version": 1}, indent=2) + "\n", encoding="utf-8")

## Bind paths, bytes, media types, and hashes

The manifest is calculated from exact file bytes. Rebuild it after every change; a stale manifest must never be presented as a valid release.

In [ ]:
def file_record(path):
    payload = path.read_bytes()
    return {
        "path": path.relative_to(workspace).as_posix(),
        "size_bytes": len(payload),
        "mime_type": mimetypes.guess_type(path.name)[0] or "application/octet-stream",
        "sha256": hashlib.sha256(payload).hexdigest(),
    }

manifest = [file_record(path) for path in sorted(workspace.rglob("*")) if path.is_file()]
manifest_bytes = json.dumps(manifest, sort_keys=True, separators=(",", ":")).encode("utf-8")
manifest_sha256 = hashlib.sha256(manifest_bytes).hexdigest()
print(json.dumps({"manifest_sha256": manifest_sha256, "files": manifest}, indent=2))

## Verify again before any upload

A publishable revision also needs required malware, secret, and format-policy scans; an applicable offline model, dataset, app, or notebook analysis; a complete release manifest; and explicit human submission. Local hashes alone do not satisfy those gates.

In [ ]:
recalculated = [file_record(path) for path in sorted(workspace.rglob("*")) if path.is_file()]
checks = {
    "manifest_matches_current_bytes": recalculated == manifest,
    "all_sha256_are_full": all(len(item["sha256"]) == 64 for item in manifest),
    "required_scans_passed": False,
    "applicable_analysis_passed": False,
    "human_submission_confirmed": False,
}
print(json.dumps(checks, indent=2))
if all(checks.values()):
    print("Ready for an authenticated publishing client.")
else:
    print("Not publishable yet: the missing gates remain explicit.")